# YOLOv11 display-screen detector training

Train a YOLOv11 detector for the digital scale display. The expected dataset format is a normal YOLO detect dataset with `data.yaml` and one class such as `display` or `screen`.

After training, the best weight can be used by `ocr_test.ipynb` to crop the display before running seven-segment OCR and PaddleOCR.

In [ ]:
from pathlib import Path
import os
import glob
import torch
from IPython.display import Image, display

try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None
    print('ultralytics is not installed in this environment.')

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
print('PROJECT_DIR =', PROJECT_DIR)
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())

In [ ]:
!nvidia-smi

In [ ]:
# Set the physical GPU visible to this notebook if needed.
# Use one GPU for the whole notebook; YOLO will see it as device 0.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Change this path to your display-screen YOLO dataset.
DATASET_DIR = Path('/data/run01/scxj083/SEED_project/display_screen_yolo_dataset')
DATA_YAML = DATASET_DIR / 'data.yaml'

MODEL = 'yolo11x.pt'
IMG_SIZE = 640
EPOCHS = 200
PATIENCE = 30
BATCH = 8
DEVICE = 0
WORKERS = 8

RUN_PROJECT = PROJECT_DIR / 'runs' / 'display_detect'
RUN_NAME = 'yolo11_display'
BEST_WEIGHTS = RUN_PROJECT / RUN_NAME / 'weights' / 'best.pt'

print('DATA_YAML =', DATA_YAML, 'exists=', DATA_YAML.exists())
print('RUN_PROJECT =', RUN_PROJECT)
print('BEST_WEIGHTS =', BEST_WEIGHTS)

In [ ]:
torch.cuda.empty_cache()
assert DATA_YAML.exists(), f'data.yaml not found: {DATA_YAML}'

In [ ]:
!yolo task=detect mode=train model={MODEL} data={DATA_YAML} epochs={EPOCHS} patience={PATIENCE} imgsz={IMG_SIZE} batch={BATCH} device={DEVICE} workers={WORKERS} project={RUN_PROJECT} name={RUN_NAME} amp=False save=True plots=True

In [ ]:
!yolo task=detect mode=val model={BEST_WEIGHTS} data={DATA_YAML} imgsz={IMG_SIZE} batch={BATCH} device={DEVICE} project={RUN_PROJECT} name={RUN_NAME}_val

In [ ]:
results_png = RUN_PROJECT / RUN_NAME / 'results.png'
if results_png.exists():
    display(Image(filename=str(results_png), height=620))
else:
    print('results.png not found yet:', results_png)

In [ ]:
TEST_IMAGES = DATASET_DIR / 'test' / 'images'
if not TEST_IMAGES.exists():
    TEST_IMAGES = DATASET_DIR / 'valid' / 'images'
print('TEST_IMAGES =', TEST_IMAGES, 'exists=', TEST_IMAGES.exists())

In [ ]:
!yolo task=detect mode=predict model={BEST_WEIGHTS} source={TEST_IMAGES} conf=0.25 imgsz={IMG_SIZE} device={DEVICE} project={RUN_PROJECT} name={RUN_NAME}_predict save=True show_labels=False show_conf=True boxes=True max_det=10

In [ ]:
preview_dir = RUN_PROJECT / f'{RUN_NAME}_predict'
preview_images = sorted(glob.glob(str(preview_dir / '*.jpg')) + glob.glob(str(preview_dir / '*.png')))
print('preview images:', len(preview_images))
for image_path in preview_images[:12]:
    display(Image(filename=image_path, height=520))

In [ ]:
print(BEST_WEIGHTS)